# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humaisali/FlyRank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Structured Content Archetype Clustering.**

I'm picking this lane because the starter slice already hints that pages don't move as one
blob — they split along more than one axis at once. Splitting visible pages on click-capture
(CTR) and on-page engagement independently produces two roughly even, genuinely different
groups rather than one dominant pattern (see the numbers below). At the same time, simple
one-off rules like "stale and visible" or "declining with demand" fire on overlapping sets of
pages — a meaningful share of the inventory trips two or more of these flags simultaneously.
That's a sign the underlying structure is a handful of recurring page *types*, not a single
score. Clustering lets me discover and name those types (e.g. a "high exposure, low engagement"
group) from evidence instead of hand-writing an ever-growing list of if/else flags. I'm not
using article text — the dataset is metrics, buckets, and token counts only — so this is
structured/metric clustering, not semantic clustering, and I'll be careful never to call it
that.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/humaisali/FlyRank-ML-Internship-Starter-Repo"
REPO_DIR = "FlyRank-ML-Internship-Starter-Repo"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Lane: Structured Content Archetype Clustering")
print("Starter dataset:", df.shape[0], "rows,", df.shape[1], "columns")
print("No free-text column present — confirms this can only be metric/structural clustering, never semantic:")
print(list(df.columns))

Lane: Structured Content Archetype Clustering
Starter dataset: 30000 rows, 44 columns
No free-text column present — confirms this can only be metric/structural clustering, never semantic:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. The question: decision, action, cost of a wrong call

**Research question:** What performance archetypes exist across the content inventory, and
which action (protect / improve / rewrite / merge / prune / monitor) fits each one?

- **Decision it improves:** which action bucket a content strategist assigns to a *group* of
  pages first, instead of reviewing thousands of pages one at a time. Archetype membership is
  the triage lens — "this page looks like the *page-one decay risk* type" is a faster, more
  consistent starting point than re-deriving a verdict from 40+ raw columns every time.
- **Who acts, and how:** an SEO/content strategist doing weekly refresh triage with limited
  review hours. They'd pull the highest-priority archetype's pages first (e.g. "visible but
  losing engagement") rather than scrolling a flat, unordered list.
- **Cost of a wrong call:** two directions. (1) A page gets mis-grouped into a "protect" or
  "champion" archetype when it's quietly decaying — it doesn't get reviewed until the traffic
  is already gone. (2) A page gets flagged for review that's actually fine — that's wasted
  editor time pulled away from a page that needed it more. Both costs are about *misallocated,
  limited human attention*, not a broken model in production.
- **Why data/ML helps:** the signals are tangled, not simple. A page can be visible **and**
  declining **and** thin **and** sitting just past its update window all at once — the code
  cell below shows how often that overlap actually happens in this slice. Untangling that by
  eye across 30,000 rows (or 519,606 in the warehouse) isn't realistic; a plain single
  if-statement can flag one pattern, but it can't summarize a page into one coherent "type" the
  way a small number of learned/described clusters can.

**One-paragraph frame:** For a content strategist deciding which action bucket (protect,
improve, rewrite, merge, prune, monitor) to assign first, we will build cluster profiles and
archetype labels from structured content and search-performance metrics (impressions, clicks,
CTR, position, engagement, freshness, word count, buckets) — there is no single predicted
target, since clustering is unsupervised. Quality will be judged by cluster separation
(silhouette) plus a by-hand sense-check that each cluster's typical numbers actually describe a
believable archetype. A wrong call costs misallocated editor hours, or a real decliner going
unreviewed while its exposure quietly erodes. A plain rule isn't enough because the same page
routinely trips multiple overlapping flags at once, and a small set of coherent archetypes
should describe the inventory more usefully than a growing list of separate if/else flags. I
will claim only observational, descriptive, decision-support results.

In [2]:
# How often do the simple rule-based patterns from the lane guide overlap on the SAME page?
# (definitions copied exactly from docs/ml-intern-dataset-and-lane-guide.md, section 5)
stale_visible          = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
declining_with_demand  = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
thin_visible           = (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)
page_one_decay         = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
low_ctr_visible        = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)

flag_count = (stale_visible.astype(int) + declining_with_demand.astype(int) + thin_visible.astype(int)
              + page_one_decay.astype(int) + low_ctr_visible.astype(int))

n = len(df)
print(f"Pages matching 2+ overlapping simple flags at once: {(flag_count >= 2).sum():,} ({(flag_count >= 2).mean()*100:.1f}%)")
print(f"Pages matching 0 of these flags:                    {(flag_count == 0).sum():,} ({(flag_count == 0).mean()*100:.1f}%)")
print()
print("-> a meaningful share of the inventory carries multiple signals simultaneously —")
print("   that's the tangled-signal case for clustering instead of one more if-statement.")

Pages matching 2+ overlapping simple flags at once: 8,438 (28.1%)
Pages matching 0 of these flags:                    10,350 (34.5%)

-> a meaningful share of the inventory carries multiple signals simultaneously —
   that's the tangled-signal case for clustering instead of one more if-statement.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter CSV, loaded above, that make this lane look worth the next 7
weeks:

1. The inventory is already structurally mixed, before any performance metric is even
   considered: 3 `content_type` values and 5 `position_tier` values, at very different sizes
   (see counts below) — different populations exist in the data on day one.
2. Among **visible** pages (`impressions_90d >= 500`, 16,726 rows / 55.8% of the slice),
   splitting on the median of two independent axes — CTR and engagement rate — produces two
   genuinely different, comparably sized groups rather than one dominant group. That's a sign
   these axes move somewhat independently, which is exactly the condition clustering exploits.
3. From Section 2's code: a meaningful share of pages trip 2+ overlapping simple flags at once
   — the same page can be visible, declining, *and* past its freshness window simultaneously.

In [3]:
print("1) Structural mix before any performance metric:")
print(df["content_type"].value_counts())
print()
print(df["position_tier"].value_counts())
print()

print("2) Visible pages (impressions_90d >= 500) split on two independent axes:")
visible = df[df["impressions_90d"] >= 500].copy()
med_ctr = visible["ctr"].median()
med_eng = visible["engagement_rate"].median()
high_ctr = visible["ctr"] >= med_ctr
high_eng = visible["engagement_rate"] >= med_eng
quad = pd.crosstab(high_ctr.map({True: "high_ctr", False: "low_ctr"}),
                    high_eng.map({True: "high_eng", False: "low_eng"}))
print(f"visible pages: {len(visible):,} ({len(visible)/len(df)*100:.1f}% of the slice)")
print(f"median ctr={med_ctr:.2f}%, median engagement_rate={med_eng:.2f}%")
print(quad)

1) Structural mix before any performance metric:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64

2) Visible pages (impressions_90d >= 500) split on two independent axes:
visible pages: 16,726 (55.8% of the slice)
median ctr=0.17%, median engagement_rate=0.00%
engagement_rate  high_eng
ctr                      
high_ctr             8474
low_ctr              8252


## 4. Careful words: what I can and can't claim

**What this work CAN say (observed / descriptive / decision-support):**
- The starter (and later warehouse) data contains groups of pages whose observed metrics
  cluster together over the measured window — described in plain, checkable terms (e.g. "this
  group has high exposure and low CTR relative to its position tier").
- Cluster membership is a **prioritization lens**: a reasonable place for a reviewer with
  limited time to start, not a verdict on any individual page.
- Patterns are **directional and observational** — I watched what happened, I did not run an
  experiment.

**What this work will NEVER claim:**
- No causal proof. Clustering (or any later refresh action) cannot claim it *caused* a page to
  recover — that needs an experiment, which this data doesn't give me.
- Not semantic clustering. There's no article text in this dataset, so clusters describe
  metric/structural behavior only — I will never call this "topic" or "meaning-based"
  clustering.
- Not a claim about Google's algorithm. I'm describing observed search/engagement behavior, not
  reverse-engineering a ranking factor.
- Not a fixed ground truth. Unsupervised clusters are a lens I chose (features, scaling, k) —
  re-running with different choices can reshape them, so I'll validate with silhouette scores
  *and* a by-hand read of whether each cluster's numbers tell a believable story, and I'll say
  so plainly in the write-up.
- IDs (`content_id`, `client_id`) are pseudonyms for grouping only, never features, and nothing
  in any output will contain a real client name, URL, or query.

In [4]:
# Confirm nothing raw/identifying is floating around, and that IDs are pseudonymous — not features.
print("Sample content_id values (pseudonyms, grouping/joins only):")
print(df["content_id"].head(3).tolist())
print()
print("Sample client_id values (pseudonyms, use for client-holdout grouping only):")
print(df["client_id"].head(3).tolist())
print()
print("Distinct clients in this slice:", df["client_id"].nunique())
print("No article-text column exists in this dataset -> confirms clustering here is structural/metric-based, never semantic.")

Sample content_id values (pseudonyms, grouping/joins only):
['content_304f48230142', 'content_a1fb4e703a9e', 'content_9aa793d4d895']

Sample client_id values (pseudonyms, use for client-holdout grouping only):
['client_f369cb89fc', 'client_4e07408562', 'client_7f2253d7e2']

Distinct clients in this slice: 32
No article-text column exists in this dataset -> confirms clustering here is structural/metric-based, never semantic.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.